In [145]:
import cv2
import mediapipe as mp
import pyautogui
import numpy as np
import random  # Fix for random function
import util  
from pynput.mouse import Button, Controller

mouse = Controller()

# Get screen size
screen_width, screen_height = pyautogui.size()

# Initialize MediaPipe Hands
mpHands = mp.solutions.hands
hands = mpHands.Hands(
    static_image_mode=False,
    model_complexity=1,
    max_num_hands=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.75
)
draw = mp.solutions.drawing_utils  # For drawing hand landmarks

def move_mouse(index_finger_tip):
    """Move the mouse pointer based on index finger tip position"""
    if index_finger_tip is not None:
        x = int(index_finger_tip.x * screen_width)
        y = int(index_finger_tip.y * screen_height)
        pyautogui.moveTo(x, y)


def find_finger_tip(processed):
    """Find the tip of the index finger from hand landmarks"""
    if processed.multi_hand_landmarks:
        hand_landmarks = processed.multi_hand_landmarks[0]
        return hand_landmarks.landmark[mpHands.HandLandmark.INDEX_FINGER_TIP]
    
    return None


def is_left_click(landmarks_list, thumb_index_dist):
    return (
        util.get_angles(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and
        util.get_angles(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 90 and
        thumb_index_dist > 50
    )


def is_right_click(landmarks_list, thumb_index_dist):
    return (
        util.get_angles(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and
        util.get_angles(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 90 and
        thumb_index_dist > 50
    )


def is_double_click(landmarks_list, thumb_index_dist):
    return (
        util.get_angles(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and
        util.get_angles(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and
        thumb_index_dist > 50
    )


def is_screenshot(landmarks_list, thumb_index_dist):
    return (
        util.get_angles(landmarks_list[5], landmarks_list[6], landmarks_list[8]) < 50 and
        util.get_angles(landmarks_list[9], landmarks_list[10], landmarks_list[12]) < 50 and
        thumb_index_dist < 50
    )


def detect_gestures(frame, landmarks_list, processed):
    """Detect gestures based on landmark positions"""
    if len(landmarks_list) >= 5:
        index_finger_tip = find_finger_tip(processed)
        thumb_index_distance = util.get_distance([landmarks_list[4], landmarks_list[5]])

        # Move mouse
        if thumb_index_distance is not None and thumb_index_distance < 50:
            if util.get_angles(landmarks_list[5], landmarks_list[6], landmarks_list[8]) > 90:
                move_mouse(index_finger_tip)

        # Left Click
        if is_left_click(landmarks_list, thumb_index_distance):
            mouse.press(Button.left)
            mouse.release(Button.left)
            cv2.putText(frame, "Left Click", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        # Right Click
        elif is_right_click(landmarks_list, thumb_index_distance):
            mouse.press(Button.right)
            mouse.release(Button.right)
            cv2.putText(frame, "Right Click", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        # Double Click
        elif is_double_click(landmarks_list, thumb_index_distance):
            pyautogui.doubleClick()
            cv2.putText(frame, "Double Click", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Screenshot
        elif is_screenshot(landmarks_list, thumb_index_distance):
            im1 = pyautogui.screenshot()
            label = random.randint(1, 1000)  # Fix random function usage
            im1.save(f'my_screenshot_{label}.png')
            cv2.putText(frame, "Screenshot Taken", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)


def main():
    cap = cv2.VideoCapture(0)

    # Optional: Make window resizable and set default size
    cv2.namedWindow('Frame', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Frame', 800, 600)

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            frameRGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            processed = hands.process(frameRGB)

            landmarks_list = []

            if processed.multi_hand_landmarks:
                for hand_landmarks in processed.multi_hand_landmarks:
                    draw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS)
                    for lm in hand_landmarks.landmark:
                        landmarks_list.append((lm.x, lm.y))

            # Detect gestures using only the first detected hand
            if processed.multi_hand_landmarks:
                detect_gestures(frame, landmarks_list, processed)

            # Resize frame before displaying
            resized_frame = cv2.resize(frame, (640, 480))
            cv2.imshow('Frame', resized_frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except Exception as e:
        print(f"Error: {e}")
    finally:
        cap.release()
        cv2.destroyAllWindows()


main()
